# 01 — Scaled Dot-Product Attention

This notebook walks through the **core building block** of the Transformer.

**Paper equation:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

from models.attention import scaled_dot_product_attention, create_causal_mask
from visualizations.attention_heatmaps import plot_attention_heatmap

torch.manual_seed(42)

## Step 1: Create Q, K, V tensors

Shapes use `(batch, n_heads, seq_len, d_k)` — the layout used inside multi-head attention.

In [ ]:
batch, n_heads, seq_len, d_k = 1, 1, 4, 8

Q = torch.randn(batch, n_heads, seq_len, d_k)
K = torch.randn(batch, n_heads, seq_len, d_k)
V = torch.randn(batch, n_heads, seq_len, d_k)

print('Q shape:', Q.shape)  # (1, 1, 4, 8)
print('K shape:', K.shape)
print('V shape:', V.shape)

## Step 2: Compute attention

Internally:
1. `scores = Q @ K^T / sqrt(d_k)` → `(batch, heads, seq_len, seq_len)`
2. `weights = softmax(scores)`
3. `output = weights @ V` → `(batch, heads, seq_len, d_k)`

In [ ]:
output, weights = scaled_dot_product_attention(Q, K, V)

print('Attention weights shape:', weights.shape)  # (1, 1, 4, 4)
print('Output shape:', output.shape)              # (1, 1, 4, 8)
print('\nWeights sum to 1 along last dim:', weights.sum(dim=-1))

## Step 3: Why scale by √d_k?

Without scaling, dot products grow with dimension → softmax saturates → tiny gradients.

In [ ]:
d_k = 512
q = torch.randn(1, 1, 1, d_k)
k = torch.randn(1, 1, 1, d_k)

raw_score = (q @ k.transpose(-2, -1)).item()
scaled_score = raw_score / (d_k ** 0.5)

print(f'Raw dot product (d_k=512):     {raw_score:.2f}')
print(f'Scaled dot product (/ sqrt d_k): {scaled_score:.2f}')

## Step 4: Causal mask (decoder)

Position *i* cannot attend to positions *j > i*.

In [ ]:
causal_mask = create_causal_mask(seq_len)
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print('Causal attention weights (head 0):')
print(masked_weights[0, 0].detach().round(decimals=3))

## Step 5: Visualize attention as a heatmap

In [ ]:
tokens = ['<sos>', 'hello', 'world', '<eos>']
w = weights[0, 0].detach().numpy()

fig = plot_attention_heatmap(w, tokens, tokens, title='Sample Attention Weights')
plt.show()